In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum as spark_sum, month, to_date

In [ ]:
spark = SparkSession.builder.appName("RetailStoreInsights").getOrCreate()

In [ ]:
df = spark.read.csv("cleaned_sales.csv", header=True, inferSchema=True)

In [ ]:
df = df.withColumn("sale_date", to_date("sale_date"))
df = df.withColumn("sale_month", month("sale_date"))

In [ ]:
underperforming = df.filter((col("quantity") < 5) | (col("profit_margin") < 10))
underperforming.show()

In [ ]:
store_monthly_revenue = df.groupBy("store_id", "sale_month").agg(
    avg("revenue").alias("avg_monthly_revenue"),
    spark_sum("revenue").alias("total_monthly_revenue")
)
store_monthly_revenue.show()

In [ ]:
underperforming.write.mode("overwrite").csv("underperforming_products", header=True)
store_monthly_revenue.write.mode("overwrite").csv("store_summary", header=True)